# 02 — Multi-Scenario Revenue Forecasting

This notebook demonstrates how to generate bear/base/bull revenue projections
using Claude and the `revenue_forecast` prompt template.

**Prerequisites:**
```bash
pip install anthropic pandas matplotlib
export ANTHROPIC_API_KEY="sk-ant-..."
```

In [ ]:
import sys, json
sys.path.insert(0, '../scripts')
from utils import load_template, fill_template, extract_user_prompt, extract_system_prompt, call_claude, extract_json

## 1. Prepare historical data

In [ ]:
historical_csv = """quarter,revenue
Q1-2024,1800
Q2-2024,1950
Q3-2024,2100
Q4-2024,2400
Q1-2025,2550
Q2-2025,2700"""

context = """
B2B fintech company focused on cross-border payments.
Strong enterprise pipeline. Recently launched lending product.
Facing increased competition from Wise and Airwallex.
EU regulatory changes (PSD3) may increase compliance costs.
"""

print(historical_csv)

## 2. Generate the forecast

In [ ]:
template = load_template('revenue_forecast')
system_prompt = extract_system_prompt(template)
user_prompt = extract_user_prompt(template)
user_prompt = fill_template(user_prompt, {
    'COMPANY_NAME': 'PayFlow',
    'HISTORICAL_REVENUE_CSV': historical_csv,
    'BUSINESS_CONTEXT': context,
    'FORECAST_QUARTERS': '4',
})

raw = call_claude(user_prompt, system_prompt=system_prompt)
forecast = extract_json(raw)
print(json.dumps(forecast, indent=2))

## 3. Visualise the scenarios

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(10, 5))

for scenario_name in ['bear', 'base', 'bull']:
    scenario = forecast.get('scenarios', {}).get(scenario_name, {})
    projections = scenario.get('projections', [])
    if projections:
        df = pd.DataFrame(projections)
        ax.plot(df['quarter'], df['revenue'], marker='o', label=scenario_name.capitalize())

ax.set_title('Revenue Forecast — Bear / Base / Bull')
ax.set_ylabel('Revenue ($M)')
ax.legend()
ax.grid(True, alpha=0.3)
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

## 4. Compare assumptions

The real value is in the **assumptions** behind each scenario — these are what you should debate with your team.

In [ ]:
for name in ['bear', 'base', 'bull']:
    scenario = forecast.get('scenarios', {}).get(name, {})
    print(f"\n{'='*40}")
    print(f"{name.upper()} CASE")
    print(f"Primary driver: {scenario.get('primary_driver', 'N/A')}")
    for a in scenario.get('assumptions', []):
        print(f"  • {a}")